# PA3 Task 1 — Kaggle runner (fine-tuning, retention, diagnostics)

This one notebook is run as **two Kaggle versions** (more only if a run needs resuming), each with
*Save Version → Save & Run All (Commit)*, so it runs in the background (≤ 12 h) and keeps `/kaggle/working` as its output.

| Version | `STAGE` | GPU 0 queue | GPU 1 queue | Main output |
|---|---|---|---|---|
| 1 | `"train"` | data → preflight → `full_ft` → `l2sp` | `lora_r4` → `lora_r16` → `oplora_r16` | `ckpts/`, `data/`, training logs |
| 2 | `"eval"`  | benchmarks + qualitative: baseline, Full FT mid/final, L2-SP | merge → benchmarks + qualitative: LoRA-4/16, OPLoRA-16 | `results/`, `merged/` |

Training stops cleanly before `MAX_HOURS` (recovery state saved). If anything is unfinished after version 1, run again with
`STAGE="train"` and `PREV_OUTPUT` = version 1's output: unfinished runs resume exactly where they stopped, finished ones are skipped.

Notebook settings: **Accelerator = GPU T4 ×2**, **Internet = On**.

In [ ]:
# ---------------- settings (edit these) ----------------
REPO_URL      = "https://github.com/wasaymj/ATML.git"
BRANCH        = "main"
STAGE         = "train"   # "train" (version 1, and any resumption) or "eval" (version 2)
PREV_OUTPUT   = ""        # e.g. "/kaggle/input/pa3-task1-runner" (Add Input -> Your Work -> this notebook's previous version)
MAX_HOURS     = 10.5      # per training process; leaves time for checkpoint I/O before Kaggle's 12 h limit
RUN_PREFLIGHT = True      # ~10-15 min of checks before training (recommended on the first run)

import os, subprocess, sys, time, shutil, glob
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA3/Task 1"
CKPT = f"{WORK}/ckpts"
MERGED = f"{WORK}/merged"
LOGS = f"{WORK}/logs"
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(shutil.disk_usage(WORK))

In [ ]:
# ---------------- get the code ----------------
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- install pinned dependencies (torch stays Kaggle's CUDA build) ----------------
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)
print(subprocess.run([sys.executable, "-c",
    "import torch, transformers, peft, datasets, importlib.metadata as m; "
    "print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, "
    "'| datasets', datasets.__version__, '| lm-eval', m.version('lm_eval'))"], capture_output=True, text=True))

In [ ]:
# ---------------- restore the previous version's outputs (resume / evaluation) ----------------
def copy_tree(src, dst):
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("restored", src, "->", dst)
if PREV_OUTPUT:
    copy_tree(f"{PREV_OUTPUT}/ckpts", CKPT)
    copy_tree(f"{PREV_OUTPUT}/merged", MERGED)
    copy_tree(f"{PREV_OUTPUT}/ATML/PA3/Task 1/data", f"{TASK}/data")
    copy_tree(f"{PREV_OUTPUT}/ATML/PA3/Task 1/results", f"{TASK}/results")
print(shutil.disk_usage(WORK))

In [ ]:
# ---------------- helpers ----------------
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")

def run(*args, gpu=None):
    # Run one step in the foreground; its log is streamed into the notebook.
    env = dict(ENV, **({"CUDA_VISIBLE_DEVICES": str(gpu)} if gpu is not None else {}))
    cmd = [sys.executable, "scripts/run_task1.py", *args, "--ckpt_root", CKPT, "--merged_root", MERGED]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=TASK, env=env).returncode

def queue(name, gpu, steps):
    # Run a list of steps one after another in the background on one GPU.
    lines = [f'cd "{TASK}"']
    for args in steps:
        lines.append(f'{sys.executable} scripts/run_task1.py {args} --ckpt_root {CKPT} --merged_root {MERGED} '
                     f'|| echo "STEP EXITED NON-ZERO: {args}"')
    script = f"{LOGS}/{name}.sh"
    open(script, "w").write("\n".join(lines) + "\n")
    log = open(f"{LOGS}/{name}.log", "a")
    print(f"[{name}] GPU {gpu}: {steps}\n   log: {LOGS}/{name}.log", flush=True)
    return subprocess.Popen(["bash", script], stdout=log, stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=10):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name, p in procs.items():
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-2:]
            print(f"--- {name} ({'running' if p.poll() is None else 'finished'})", *tail, sep="\n   ", flush=True)
        print(shutil.disk_usage(WORK), flush=True)
    print({n: p.returncode for n, p in procs.items()})

## Version 1 (`STAGE="train"`): data, preflight, and the five training runs
Preflight checks the non-thinking chat template, lm-eval split sizes (1,319 / 10,042 / 3,270 / 2,376), runs an
8-document lm-eval pass on the baseline, a forward/backward on the two longest training examples for Full FT, LoRA-16 and
OPLoRA-16 (memory), the matched LoRA-16/OPLoRA-16 initialization, and an OPLoRA merge round trip.

In [ ]:
if STAGE == "train":
    assert run("--step", "data", gpu=0) == 0           # MetaMathQA 40k (+ token cache) and the activation probe set
    if RUN_PREFLIGHT:
        assert run("--step", "preflight", gpu=0) == 0, "Preflight failed - read the log above before training."

In [ ]:
if STAGE == "train":
    mh = f"--max_hours {MAX_HOURS}"
    procs = {
        "gpu0_full": queue("gpu0_full", 0, [f"--step train --method full_ft {mh}", f"--step train --method l2sp {mh}"]),
        "gpu1_adapters": queue("gpu1_adapters", 1, [f"--step train --method lora_r4 {mh}",
                                                    f"--step train --method lora_r16 {mh}",
                                                    f"--step train --method oplora_r16 {mh}"]),
    }
    wait(procs)

`--max_hours` applies per process, so in a queue a long first job can leave later jobs paused or unstarted.
`status` shows exactly what is finished or resumable.

In [ ]:
run("--step", "status")
run("--step", "collect")      # copies the small training logs into results/training/

## Version 2 (`STAGE="eval"`): merge, benchmarks (7 conditions), qualitative, diagnostics, report

In [ ]:
if STAGE == "eval":
    procs = {
        "gpu0_eval": queue("gpu0_eval", 0, ["--step benchmarks --conditions baseline,full_ft_mid,full_ft,l2sp",
                                            "--step qualitative --conditions baseline,full_ft_mid,full_ft,l2sp"]),
        "gpu1_eval": queue("gpu1_eval", 1, ["--step merge",
                                            "--step benchmarks --conditions lora_r4,lora_r16,oplora_r16",
                                            "--step qualitative --conditions lora_r4,lora_r16,oplora_r16"]),
    }
    wait(procs)
    run("--step", "benchmarks", gpu=0)    # finished conditions are cached; writes all_conditions_summary.json
    run("--step", "qualitative", gpu=0)   # groups the 168 responses and writes the 42-row annotation sheet
    run("--step", "diagnostics", gpu=0)
    run("--step", "report")
    run("--step", "status")

In [ ]:
# ---------------- package the small, git-tracked outputs for download ----------------
shutil.make_archive(f"{WORK}/task1_results", "zip", TASK, "results")
shutil.make_archive(f"{WORK}/task1_data_ids", "zip", TASK, "data")
print(sorted(os.listdir(WORK)))

In [ ]:
# Inline preview of the tables and figures (after the eval version)
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/*.md")):
    display(Markdown(f"### {os.path.basename(t)}\n" + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/*.png")):
    display(Image(fig, width=700))